Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
# DBTITLE 1,Cell 2

import trino
import pandas as pd

query = """

SELECT
    ticket_date                 AS `Date`,
    date_add(ticket_date, 1)    AS `Audit Date`,
    SUM(is_connected)           AS `Connects`
FROM corp_analytics.gi_genai_ticket_level
WHERE ticket_date >= DATE '2026-05-01'
  AND ticket_date <= current_date()
GROUP BY ticket_date
ORDER BY ticket_date DESC

"""

df  = spark.sql(query).toPandas()
if not df.empty:
    display(df)

In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/1jwqtHs6pJY2APFPunyZRHXyM0o9XSHrM4fTUmX-2Ph0/edit?gid=857594292#gid=857594292"
worksheet_name = "Connected Call Count"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=1
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=1
)